In [1]:
from datetime import datetime
from pathlib import Path
import pandas as pd
import sys, os

from src.datasets import build_dataloaders
from src.features import (
    build_code_to_cities,
    build_final_dataset,
    build_rq_codebook,
    create_trip_sequences,
    train_word2vec,
)
from src.models import RQKmeansPredictor, RQVAETransformer
from src.training.code_predict import predict_top4_with_codebook, train_code_transformer
from src.utils import top_city_ids_from_train

In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd()
if not (_repo / "src").is_dir():
    _repo = _repo.parent
sys.path.insert(0, str(_repo))

from src.utils.paths import data_dir, output_dir as repo_output_dir

output_dir = repo_output_dir()
train_set = pd.read_csv(data_dir() / "train_set.csv")
test_set = pd.read_csv(data_dir() / "test_set.csv")

In [3]:
import pandas as pd
import numpy as np

def create_enhanced_sequences(df):
    # 1. 时间格式转换与排序
    df['checkin'] = pd.to_datetime(df['checkin'])
    df['checkout'] = pd.to_datetime(df['checkout'])
    df = df.sort_values(['utrip_id', 'checkin'])
    
    # 2. 特征提取
    # 计算停留天数 (Stay Duration)
    df['stay_duration'] = (df['checkout'] - df['checkin']).dt.days
    # 剪裁异常值（比如住超过30天的极少数情况），统一到 1-30 范围内
    df['stay_duration'] = df['stay_duration'].clip(1, 30)
    
    # 提取月份 (Month) - 捕捉季节性
    df['checkin_month'] = df['checkin'].dt.month
    
    # 3. 聚合行程
    # 我们不仅要 city_id 序列，还要 stay_duration 序列
    sequences = df.groupby('utrip_id').agg({
        'city_id': list,
        'stay_duration': list,      # 新增：停留时长序列
        'checkin_month': 'first',   # 新增：行程开始月份
        'booker_country': 'first',  # 用户背景
        'device_class': 'first'     # 用户背景
    }).reset_index()
    
    return sequences

# 应用增强版预处理
print("正在提取增强特征 (Stay Duration, Month, etc.)...")
train_trips = create_enhanced_sequences(train_set)
test_trips = create_enhanced_sequences(test_set)

正在提取增强特征 (Stay Duration, Month, etc.)...


In [4]:
# 1. 处理国家编码 (Booker Country)
all_countries = sorted(train_set['booker_country'].unique())
country_to_idx = {country: i for i, country in enumerate(all_countries)}

# 2. 处理设备编码 (Device Class)
all_devices = sorted(train_set['device_class'].unique())
device_to_idx = {device: i for i, device in enumerate(all_devices)}

# 获取总类别数，用于定义 Embedding 层
num_countries = len(all_countries)
num_devices = len(all_devices)

In [13]:
# train_trips = create_trip_sequences(train_set)
# test_trips = create_trip_sequences(test_set)

In [14]:
train_trips

,utrip_id,city_id,stay_duration,checkin_month,booker_country,device_class
0,1000027_1,"[8183, 15626, 60902, 30628]","[1, 2, 2, 3]",8,Elbonia,desktop
1,1000033_1,"[38677, 52089, 21328, 27485, 38677]","[2, 1, 2, 2, 3]",4,Gondal,mobile
2,1000045_1,"[64876, 55128, 9608, 31817, 36170, 58178, 36063]","[2, 2, 2, 1, 1, 2, 1]",6,The Devilfire Empire,desktop
3,1000083_1,"[55990, 14705, 35160, 36063]","[1, 1, 1, 2]",6,The Devilfire Empire,mobile
4,100008_1,"[11306, 12096, 6761, 6779, 65690]","[3, 1, 1, 2, 2]",7,Gondal,desktop
...,...,...,...,...,...,...
217681,999776_1,"[17775, 66634, 17775, 17775]","[1, 1, 1, 1]",3,Gondal,desktop
217682,999839_1,"[8335, 21328, 8335, 48968]","[2, 1, 3, 2]",8,The Devilfire Empire,mobile
217683,999842_1,"[51291, 66969, 67169, 24036]","[1, 1, 1, 1]",5,Gondal,desktop
217684,999855_1,"[382, 38509, 18930, 38509, 51145, 11179, 61881...","[5, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 2, 1]",4,Gondal,desktop


In [5]:
w2v = train_word2vec(train_trips, vector_size=128, window=5)
# 调用方法：
# vec = w2v.wv[str(55990)]
# print(vec)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [6]:
city_to_codes = build_rq_codebook(train_set, w2v, n_clusters=128)

In [8]:
import pandas as pd
import numpy as np

def debug_rq_codes(city_to_codes, train_set):
    print("--- 🔍 RQ-VAE 编码质量排查报告 ---")
    
    # 1. 碰撞率分析 (Collision Analysis)
    code_series = pd.Series(list(city_to_codes.values()))
    counts = code_series.value_counts()
    
    num_cities = len(city_to_codes)
    num_active_codes = len(counts)
    possible_codes = 128 * 128 # 1024
    
    print(f"1️⃣ 坑位占用情况:")
    print(f"   - 总城市数: {num_cities}")
    print(f"   - 实际激活的代码组合数: {num_active_codes} / {possible_codes}")
    print(f"   - 平均每个代码负担的城市数: {num_cities / num_active_codes:.2f}")
    print(f"   - 最拥挤的‘死胡同’ (Top 1代码包含城市数): {counts.max()}")
    print(f"   - 只有1个城市的‘精准代码’占比: {(counts == 1).sum() / num_active_codes:.2%}")
    
    if counts.max() > 100:
        print("   ⚠️ 警告: 碰撞极其严重！模型预测出这个代码后，等于从100多个城市里盲猜。")

    # 2. 覆盖度分析
    print(f"\n2️⃣ 流行度覆盖分析:")
    # 看看最热门的城市是否被分开了
    top_20_cities = train_set['city_id'].value_counts().head(20).index.tolist()
    top_city_codes = [city_to_codes.get(c) for c in top_20_cities if c in city_to_codes]
    unique_top_codes = len(set(top_city_codes))
    
    print(f"   - 最热门的20个城市占用了 {unique_top_codes} 个独立代码")
    if unique_top_codes < 15:
        print("   ⚠️ 警告: 热门城市高度重合。模型分不清这些最常去的目的地。")

    # 3. 训练集中的路径模糊度 (Target Entropy)
    # 看看在训练集里，是不是有很多不同的目标城市对应同一个代码对
    print(f"\n3️⃣ 语义模糊度预测:")
    # 这里随机抽取几个城市看看它们属于哪个大类 (code1)
    sample_cities = train_set['city_id'].sample(5).tolist()
    for c in sample_cities:
        code = city_to_codes.get(c)
        print(f"   - 城市 {c} 映射为 -> {code}")

# 运行排查
debug_rq_codes(city_to_codes, train_set)

--- 🔍 RQ-VAE 编码质量排查报告 ---
1️⃣ 坑位占用情况:
   - 总城市数: 39901
   - 实际激活的代码组合数: 6521 / 16384
   - 平均每个代码负担的城市数: 6.12
   - 最拥挤的‘死胡同’ (Top 1代码包含城市数): 275
   - 只有1个城市的‘精准代码’占比: 32.59%
   ⚠️ 警告: 碰撞极其严重！模型预测出这个代码后，等于从100多个城市里盲猜。

2️⃣ 流行度覆盖分析:
   - 最热门的20个城市占用了 18 个独立代码

3️⃣ 语义模糊度预测:
   - 城市 28829 映射为 -> (45, 26)
   - 城市 47527 映射为 -> (81, 103)
   - 城市 2023 映射为 -> (41, 68)
   - 城市 34903 映射为 -> (112, 59)
   - 城市 38209 映射为 -> (78, 3)


In [9]:
def build_multi_feature_dataset(trip_df, mapping, country_map, device_map, is_test=False):
    dataset = []
    
    for _, row in trip_df.iterrows():
        cities = row['city_id']
        durations = row['stay_duration']
        
        # 1. 城市编码序列 (RQ-VAE)
        code_seq = []
        for c in cities:
            code_seq.extend(list(mapping.get(c, (0, 0))))
        
        # 2. 停留时长序列 (需要和 Code 序列长度对应)
        # 每一个 city 对应 2 个 code，所以 duration 也要复制两份，或者只在每个 city 的第一个 code 处加入
        # 这里建议简单化：每一站对应一个 duration
        dur_seq = durations
        
        # 3. 静态特征
        b_country = country_map.get(row['booker_country'], 0)
        d_class = device_map.get(row['device_class'], 0)
        month = row['checkin_month'] - 1 # 转为 0-11
        
        if is_test:
            # 同样去掉最后一站的占位符信息
            feature_dict = {
                'code_seq': code_seq[:-2],
                'dur_seq': dur_seq[:-1], # 对应前 N-1 站的时长
                'static_feats': [b_country, d_class, month]
            }
            dataset.append(feature_dict)
        else:
            if len(code_seq) >= 4:
                feature_dict = {
                    'code_seq': code_seq[:-2],
                    'dur_seq': dur_seq[:-1],
                    'static_feats': [b_country, d_class, month],
                    'target': code_seq[-2:] # 最后一站的 (c1, c2)
                }
                dataset.append(feature_dict)
                
    return dataset

multi_train_data = build_multi_feature_dataset(train_trips, city_to_codes, country_to_idx, device_to_idx)
multi_test_data = build_multi_feature_dataset(test_trips, city_to_codes, country_to_idx, device_to_idx, is_test=True)

In [20]:
# train_x, train_y = build_final_dataset(train_trips, city_to_codes, is_test=False)
# test_x, _ = build_final_dataset(test_trips, city_to_codes, is_test=True)

In [21]:
train_loader, test_loader = build_dataloaders(
    train_x, train_y, test_x, batch_size=256, pad_token=128
)

NameError: name 'train_x' is not defined

In [ ]:
model = RQVAETransformer()
model = train_code_transformer(model, train_loader, epochs=5, lr=0.001)

code_to_cities = build_code_to_cities(city_to_codes, train_set)
fallback = top_city_ids_from_train(train_set, k=4)
all_predictions = predict_top4_with_codebook(
    model, test_loader, code_to_cities, codebook_size=128, top_global=fallback
)